In [1]:
from __future__ import print_function, division

# MIMIC IIIv14 on postgres 9.4
# import os, psycopg2, re, sys, time, numpy as np, pandas as pd
import os, re, sys, time, numpy as np, pandas as pd
from sklearn import metrics
from datetime import datetime
from datetime import timedelta

from os.path import isfile, isdir, splitext
import argparse
import pickle as cPickle
import numpy.random as npr

import spacy
# TODO(mmd): Upgrade to python 3 and use scispacy (requires python 3.6)
# import scispacy

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from datapackage_io_util import (
    load_datapackage_schema,
    load_sanitized_df_from_csv,
    save_sanitized_df_to_csv,
    sanitize_df,
)
# from heuristic_sentence_splitter import sent_tokenize_rules
# from mimic_querier import *

CURRENT_DIR = "."
SQL_DIR = os.path.join(CURRENT_DIR, 'SQL_Queries')
STATICS_QUERY_PATH = os.path.join(SQL_DIR, 'statics.sql')
CODES_QUERY_PATH = os.path.join(SQL_DIR, 'codes.sql')
NOTES_QUERY_PATH = os.path.join(SQL_DIR, 'notes.sql')

In [2]:
# Output filenames
static_filename = 'static_data.csv'
static_columns_filename = 'static_colnames.txt'

dynamic_filename = 'vitals_hourly_data.csv'
columns_filename = 'vitals_colnames.txt'
subjects_filename = 'subjects.npy'
times_filename = 'fenceposts.npy'
dynamic_hd5_filename = 'vitals_hourly_data.h5'
dynamic_hd5_filt_filename = 'all_hourly_data.h5'

codes_hd5_filename = 'C.h5'
notes_hd5_filename = 'notes.hdf' # N.h5
idx_hd5_filename = 'C_idx.h5'

outcome_filename = 'outcomes_hourly_data.csv'
outcome_hd5_filename = 'outcomes_hourly_data.h5'
outcome_columns_filename = 'outcomes_colnames.txt'
outPath = './output'

# SQL command params

ID_COLS = ['subject_id', 'hadm_id', 'stay_id']
ITEM_COLS = ['itemid', 'label']
exclusion_criteria_template_vars = {}

In [3]:
def add_outcome_indicators(out_gb):
    subject_id = out_gb['subject_id'].unique()[0]
    hadm_id = out_gb['hadm_id'].unique()[0]
    stay_id = out_gb['stay_id'].unique()[0]
    max_hrs = out_gb['max_hours'].unique()[0]
    on_hrs = set()

    for index, row in out_gb.iterrows():
        on_hrs.update(range(row['starttime'], row['endtime'] + 1))

    off_hrs = set(range(max_hrs + 1)) - on_hrs
    on_vals = [0]*len(off_hrs) + [1]*len(on_hrs)
    hours = list(off_hrs) + list(on_hrs)
    return pd.DataFrame({'subject_id': subject_id, 'hadm_id':hadm_id,
                        'hours_in':hours, 'on':on_vals}) #stay_id': stay_id})


def add_blank_indicators(out_gb):
    subject_id = out_gb['subject_id'].unique()[0]
    hadm_id = out_gb['hadm_id'].unique()[0]
    #stay_id = out_gb['stay_id'].unique()[0]
    max_hrs = out_gb['max_hours'].unique()[0]

    hrs = range(max_hrs + 1)
    vals = list([0]*len(hrs))
    return pd.DataFrame({'subject_id': subject_id, 'hadm_id':hadm_id,
                        'hours_in':hrs, 'on':vals})#'stay_id': stay_id,

def continuous_outcome_processing(out_data, data, icustay_timediff):
    """

    Args
    ----
    out_data : pd.DataFrame
        index=None
        Contains subset of stay_id corresp to specific sessions where outcome observed.
    data : pd.DataFrame
        index=stay_id
        Contains full population of static demographic data

    Returns
    -------
    out_data : pd.DataFrame
    """
    print(out_data.head())
    print(data.head())
    out_data['intime'] = out_data['stay_id'].map(data['intime'].to_dict())
    out_data['outtime'] = out_data['stay_id'].map(data['outtime'].to_dict())
    out_data['max_hours'] = out_data['stay_id'].map(icustay_timediff)
    out_data['starttime'] = out_data['starttime'] - out_data['intime']
    out_data['starttime'] = out_data.starttime.apply(lambda x: x.days*24 + x.seconds//3600)
    out_data['endtime'] = out_data['endtime'] - out_data['intime']
    out_data['endtime'] = out_data.endtime.apply(lambda x: x.days*24 + x.seconds//3600)
    out_data = out_data.groupby(['stay_id'])

    return out_data
#
def fill_missing_times(df_by_sid_hid_itemid):
    max_hour = df_by_sid_hid_itemid.index.get_level_values(max_hours)[0]
    missing_hours = list(set(range(max_hour+1)) - set(df_by_sid_hid_itemid['hours_in'].unique()))
    # Add rows
    sid = df_by_sid_hid_itemid.subject_id.unique()[0]
    hid = df_by_sid_hid_itemid.hadm_id.unique()[0]
    stay_id = df_by_sid_hid_itemid.stay_id.unique()[0]
    itemid = df_by_sid_hid_itemid.itemid.unique()[0]
    filler = pd.DataFrame({'subject_id':[sid]*len(missing_hours),
                           'hadm_id':[hid]*len(missing_hours),
                           'stay_id':[stay_id]*len(missing_hours),
                           'itemid':[itemid]*len(missing_hours),
                           'hours_in':missing_hours,
                           'value':[np.nan]*len(missing_hours),
                            'max_hours': [max_hour]*len(missing_hours)})
    return pd.concat([df_by_sid_hid_itemid, filler], axis=0)

def save_pop(
        data_df, outPath, static_filename, pop_size_int,
        static_data_schema, host=None
    ):
    # Connect to local postgres version of mimic

    # Serialize to disk
    csv_fpath = os.path.join(outPath, static_filename)
    save_sanitized_df_to_csv(csv_fpath, data_df, static_data_schema)

    return data_df

# From Dave's approach!
def get_variable_mapping(mimic_mapping_filename):
    # Read in the second level mapping of the itemids
    var_map = pd.read_csv(mimic_mapping_filename, index_col=None)
    # var_map = var_map.loc[(var_map['LEVEL2'] != '') & (var_map['COUNT']>0)]
    # var_map = var_map.loc[(var_map['STATUS'] == 'ready')]
    # var_map['ITEMID'] = var_map['ITEMID'].astype(int)
    
    var_map = var_map.dropna(subset=['itemid'])
    var_map = var_map.loc[(var_map['itemid'] != '') & (var_map['count']>0)]
    var_map['ITEMID'] = var_map['itemid'].astype(int)

    return var_map

def get_variable_ranges(range_filename):
    # Read in the second level mapping of the itemid, and take those values out
    columns = [ 'LEVEL2', 'OUTLIER LOW', 'VALID LOW', 'IMPUTE', 'VALID HIGH', 'OUTLIER HIGH' ]
    to_rename = dict(zip(columns, [ c.replace(' ', '_') for c in columns ]))
    to_rename['LEVEL2'] = 'VARIABLE'
    var_ranges = pd.read_csv(range_filename, index_col=None)
    var_ranges = var_ranges[columns]
    var_ranges.rename(columns=to_rename, inplace=True)
    var_ranges = var_ranges.drop_duplicates(subset='VARIABLE', keep='first')
    var_ranges['VARIABLE'] = var_ranges['VARIABLE'].str.lower()
    var_ranges.set_index('VARIABLE', inplace=True)
    var_ranges = var_ranges.loc[var_ranges.notnull().all(axis=1)]

    return var_ranges

UNIT_CONVERSIONS = [
    ('weight',                   'oz',  None,             lambda x: x/16.*0.45359237),
    ('weight',                   'lbs', None,             lambda x: x*0.45359237),
    ('fraction inspired oxygen', None,  lambda x: x > 1,  lambda x: x/100.),
    ('oxygen saturation',        None,  lambda x: x <= 1, lambda x: x*100.),
    ('temperature',              'f',   lambda x: x > 79, lambda x: (x - 32) * 5./9),
    ('height',                   'in',  None,             lambda x: x*2.54),
]

def get_values_by_name_from_df_column_or_index(data_df, colname):
    """ Easily get values for named field, whether a column or an index

    Returns
    -------
    values : 1D array
    """
    try:
        values = data_df[colname]
    except KeyError as e:
        if colname in data_df.index.names:
            values = data_df.index.get_level_values(colname)
        else:
            raise e
    return values

def standardize_units(X, name_col='itemid', unit_col='valueuom', value_col='value', inplace=True):
    if not inplace: X = X.copy()
    name_col_vals = get_values_by_name_from_df_column_or_index(X, name_col)
    unit_col_vals = get_values_by_name_from_df_column_or_index(X, unit_col)

    try:
        name_col_vals = name_col_vals.str
        unit_col_vals = unit_col_vals.str
    except:
        print("Can't call *.str")
        print(name_col_vals)
        print(unit_col_vals)
        raise

    #name_filter, unit_filter = [
    #    (lambda n: col.contains(n, case=False, na=False)) for col in (name_col_vals, unit_col_vals)
    #]
    # TODO(mmd): Why does the above not work, but the below does?
    name_filter = lambda n: name_col_vals.contains(n, case=False, na=False)
    unit_filter = lambda n: unit_col_vals.contains(n, case=False, na=False)

    for name, unit, rng_check_fn, convert_fn in UNIT_CONVERSIONS:
        name_filter_idx = name_filter(name)
        needs_conversion_filter_idx = name_filter_idx & False

        if unit is not None: needs_conversion_filter_idx |= name_filter(unit) | unit_filter(unit)
        if rng_check_fn is not None: needs_conversion_filter_idx |= rng_check_fn(X[value_col])

        idx = name_filter_idx & needs_conversion_filter_idx

        X.loc[idx, value_col] = convert_fn(X[value_col][idx])

    return X

def range_unnest(df, col, out_col_name=None, reset_index=False):
    assert len(df.index.names) == 1, "Does not support multi-index."
    if out_col_name is None: out_col_name = col

    col_flat = pd.DataFrame(
        [[i, x] for i, y in df[col].iteritems() for x in range(y+1)],
        columns=[df.index.names[0], out_col_name]
    )

    if not reset_index: col_flat = col_flat.set_index(df.index.names[0])
    return col_flat

# TODO(mmd): improve args
def save_numerics(
    data, X, I, var_map, var_ranges, outPath, dynamic_filename, columns_filename, subjects_filename,
    times_filename, dynamic_hd5_filename, apply_var_limit, min_percent
):
    assert len(data) > 0 and len(X) > 0, "Must provide some input data to process."

    print('var map', var_map.head())
    # var_map = var_map[
    #     ['LEVEL2', 'ITEMID', 'LEVEL1']
    # ].rename(
    #     columns={'LEVEL2': 'LEVEL2', 'LEVEL1': 'LEVEL1', 'ITEMID': 'itemid'},
    # ).set_index('itemid')
    var_map = var_map.groupby('itemid').last()
    # var_map = var_map.set_index('itemid')


    print('1',X.head())


    X['value'] = pd.to_numeric(X['value'], 'coerce')
    # X.astype({k: int for k in ID_COLS if k in X.columns.tolist()}, inplace=True)
    for col in ID_COLS:
        if col in X.columns.tolist():
            X.loc[:,col] = X[col].astype(int)

    to_hours = lambda x: max(0, x.days*24 + x.seconds // 3600)

    print("joining with intime")
    print('data', data.head(), data.columns.tolist())
    X = X.set_index('stay_id').join(data[['intime']]) # left join



    charttime =X['charttime'].astype(int) / 10**9
    intime = X['intime'].astype(int) / 10**9

    hours_in = (charttime - intime)//3600
    X['hours_in'] = hours_in


    # try:
    #     from pandarallel import pandarallel
    #     pandarallel.initialize(progress_bar=True)
    #     X['hours_in'] = (X['charttime'] - X['intime']).parallel_apply(to_hours)
    # except ModuleNotFoundError:
    #     print("Install pandarallel for faster processing.")
    #     X['hours_in'] = (X['charttime'] - X['intime']).apply(to_hours)

    X.drop(columns=['charttime', 'intime'], inplace=True)
    X.set_index('itemid', append=True, inplace=True)

    print('2',X.head())

    print(len(var_map), len(var_map.groupby('itemid').first()))

    # Pandas has a bug with the below for small X
    #X = X.join([var_map, I]).set_index(['label', 'LEVEL1', 'LEVEL2'], append=True)
    X = X.join(var_map)
    print(sorted(X.columns.tolist()), sorted(I.columns.tolist()))
    X = X.join(I,  rsuffix='_remove').set_index(['label'], append=True) # join on label, keep left where columns overlap

    print('3',X.head())

    # standardize_units(X, name_col='LEVEL1', inplace=True)
    standardize_units(X, name_col='label', inplace=True)

    if apply_var_limit > 0: 
        # X = apply_variable_limits(X, var_ranges, 'LEVEL2')
        X = apply_variable_limits(X, var_ranges, 'label')

    print('4',X.head())

    # group_item_cols = ['LEVEL2'] if group_by_level2 else ITEM_COLS
    group_item_cols = ITEM_COLS
    # drop count and ITEMID
    X = X.drop(columns=['count', 'ITEMID'])
    X = X.groupby(ID_COLS + ITEM_COLS + ['hours_in']).agg(['mean', 'std', 'count'])
    print('after groupby',X.head())
    X.columns = X.columns.droplevel(0)
    X.columns.names = ['Aggregation Function']

    print('5',X.head())

    data['max_hours'] = (data['outtime'] - data['intime']).apply(to_hours)

    # TODO(mmd): Maybe can just create the index directly?
    missing_hours_fill = range_unnest(data, 'max_hours', out_col_name='hours_in', reset_index=True)
    missing_hours_fill['tmp'] = np.NaN

    # TODO(mmd): The below is a bit wasteful.
    #itemids = var_map.join(I['label']).reset_index()[group_item_cols].drop_duplicates()
    #itemids['tmp'] = np.NaN

    #missing_hours_fill = missing_hours_fill.merge(itemids, on='tmp', how='outer')

    fill_df = data.reset_index()[ID_COLS].join(missing_hours_fill.set_index('stay_id'), on='stay_id')
    fill_df.set_index(ID_COLS + ['hours_in'], inplace=True)

    # Pivot table droups NaN columns so you lose any uniformly NaN.
    X = X.unstack(level = group_item_cols)
    X.columns = X.columns.reorder_levels(order=group_item_cols + ['Aggregation Function'])

    print('6',X.head())
   

    #X = X.reset_index().pivot_table(index=ID_COLS + ['hours_in'], columns=group_item_cols, values=X.columns)
    X = X.reindex(fill_df.index)

    #X.columns = X.columns.droplevel(0).reorder_levels(order=[1, 0])
    #if group_by_level2:
    #    X.columns.names = ['LEVEL2', 'Aggregation Function'] # Won't work with ungrouped!
    #else:
    #    X.columns.names = ['itemid', 'Aggregation Function']
    #    X.columms = X.MultiIndex.from_frame(X[ITEM_COLS])

    X = X.sort_index(axis=0).sort_index(axis=1)

    print("Shape of X : ", X.shape)

    # Turn back into columns
    if columns_filename is not None:
        col_names  = [str(x) for x in X.columns.values]
        with open(os.path.join(outPath, columns_filename), 'w') as f: f.write('\n'.join(col_names))

    # Get the max time for each of the subjects so we can reconstruct!
    if subjects_filename is not None:
        np.save(os.path.join(outPath, subjects_filename), data['subject_id'].as_matrix())
    if times_filename is not None: 
        np.save(os.path.join(outPath, times_filename), data['max_hours'].as_matrix())

    #fix nan in count to beas_matrix zero
    idx = pd.IndexSlice
    # if group_by_level2:
    #     X.loc[:, idx[:, 'count']] = X.loc[:, idx[:, 'count']].fillna(0)
    # else:
    #     X.loc[:, idx[:,:,:,:, 'count']] = X.loc[:, idx[:,:,:,:, 'count']].fillna(0)
    print(X.head())
    # X.loc[:, idx[:,:,:,:, 'count']] = X.loc[:, idx[:,:,:,:, 'count']].fillna(0)
    X.loc[:, idx[:,:, 'count']] = X.loc[:, idx[:,:, 'count']].fillna(0)
    
    # Drop columns that have very few recordings
    n = round((1-min_percent/100.0)*X.shape[0])
    drop_col = []
    for k in X.columns:
        if k[-1] == 'mean':
            if X[k].isnull().sum() > n:
                drop_col.append(k[:-1])
    X = X.drop(columns = drop_col)

    ########
    if dynamic_filename is not None: np.save(os.path.join(outPath, dynamic_filename), X.as_matrix())
    if dynamic_hd5_filename is not None: X.to_hdf(os.path.join(outPath, dynamic_hd5_filename), 'X')

    return X

def save_notes(notes, outPath=None, notes_h5_filename=None):
    notes_id_cols = list(set(ID_COLS).intersection(notes.columns))# + ['row_id'] TODO: what is row_id?
    notes_metadata_cols = ['chartdate', 'charttime', 'category', 'description']

    notes.set_index(notes_id_cols + notes_metadata_cols, inplace=True)
    # preprocessing!!
    # TODO(Scispacy)
    # TODO(improve)
    # TODO(spell checking)
    # TODO(CUIs)
    # TODO This takes forever. At the very least add a progress bar.

    def sbd_component(doc):
        for i, token in enumerate(doc[:-2]):
            # define sentence start if period + titlecase token
            if token.text == '.' and doc[i+1].is_title:
                doc[i+1].sent_start = True
            if token.text == '-' and doc[i+1].text != '-':
                doc[i+1].sent_start = True
        return doc

    #convert de-identification text into one token
    def fix_deid_tokens(text, processed_text):
        deid_regex  = r"\[\*\*.{0,15}.*?\*\*\]" 
        indexes = [m.span() for m in re.finditer(deid_regex,text,flags=re.IGNORECASE)]
        for start,end in indexes:
            processed_text.merge(start_idx=start,end_idx=end)
        return processed_text

    nlp = spacy.load('en_core_web_sm') # Maybe try lg model?
    nlp.add_pipe(sbd_component, before='parser')  # insert before the parser
    disabled = nlp.disable_pipes('ner')

    def process_sections_helper(section, note, processed_sections):
        processed_section = nlp(section['sections'])
        processed_section = fix_deid_tokens(section['sections'], processed_section)
        processed_sections.append(processed_section)

    def process_note_willie_spacy(note):
        note_sections = sent_tokenize_rules(note)
        processed_sections = []
        section_frame = pd.DataFrame({'sections':note_sections})
        section_frame.apply(process_sections_helper, args=(note,processed_sections,), axis=1)
        return processed_sections

    def text_process(sent, note):
        sent_text = sent['sents'].text
        if len(sent_text) > 0 and sent_text.strip() != '\n':
            if '\n'in sent_text:
                sent_text = sent_text.replace('\n', ' ')
            note['text'] += sent_text + '\n'  

    def get_sentences(processed_section, note):
        sent_frame = pd.DataFrame({'sents': list(processed_section['sections'].sents)})
        sent_frame.apply(text_process, args=(note,), axis=1)

    def process_frame_text(note):
        try:
            note_text = str(note['text'])
            note['text'] = ''
            processed_sections = process_note_willie_spacy(note_text)
            ps = {'sections': processed_sections}
            ps = pd.DataFrame(ps)

            ps.apply(get_sentences, args=(note,), axis=1)

            return note 
        except Exception as e:
            print('error', e)
            #raise e

    notes = notes.apply(process_frame_text, axis=1)

    if outPath is not None and notes_h5_filename is not None:
        notes.to_hdf(os.path.join(outPath, notes_h5_filename), 'notes')
    return notes

def save_icd_codes(codes, outPath, codes_h5_filename):
    codes.set_index(ID_COLS, inplace=True)
    codes.to_hdf(os.path.join(outPath, codes_h5_filename), 'C')
    return codes

def save_outcome(
    data, querier, outPath, outcome_filename, outcome_hd5_filename,
    outcome_columns_filename, outcome_schema, host=None
):
    """ Retrieve outcomes from DB and save to disk

    Vent and vaso are both there already - so pull the start and stop times from there! :)

    Returns
    -------
    Y : Pandas dataframe
        Obeys the outcomes data spec
    """
    icuids_to_keep = get_values_by_name_from_df_column_or_index(data, 'stay_id')
    icuids_to_keep = set([str(s) for s in icuids_to_keep])

    # Add a new column called intime so that we can easily subtract it off
    data = data.reset_index()
    data = data.set_index('stay_id')
    data['intime'] = pd.to_datetime(data['intime']) #, format="%m/%d/%Y"))
    data['outtime'] = pd.to_datetime(data['outtime'])
    icustay_timediff_tmp = data['outtime'] - data['intime']
    icustay_timediff = pd.Series([timediff.days*24 + timediff.seconds//3600
                                  for timediff in icustay_timediff_tmp], index=data.index.values)
    # query = """
    # select i.subject_id, i.hadm_id, v.stay_id, v.ventnum, v.starttime, v.endtime
    # FROM icustay_detail i
    # INNER JOIN ventilation_durations v ON i.stay_id = v.stay_id
    # where v.stay_id in ({icuids})
    # and v.starttime between intime and outtime
    # and v.endtime between intime and outtime;
    # """

    query = """
    select i.subject_id, i.hadm_id, v.stay_id, v.starttime, v.endtime
    FROM icustay_detail i
    INNER JOIN ventilation v ON i.stay_id = v.stay_id
    where v.stay_id in ({icuids})
    and v.starttime between i.icu_intime and i.icu_outtime
    and v.endtime between i.icu_intime and i.icu_outtime;
    """

    old_template_vars = querier.exclusion_criteria_template_vars
    querier.exclusion_criteria_template_vars = dict(icuids=','.join(icuids_to_keep))

    vent_data = querier.query(query_string=query)
    # get the ventnum for mimiciv
    vent_data.loc[:, 'ventnum'] = vent_data.groupby(['stay_id']).cumcount()

    vent_data = continuous_outcome_processing(vent_data, data, icustay_timediff)
    vent_data = vent_data.apply(add_outcome_indicators)
    vent_data.rename(columns = {'on':'vent'}, inplace=True)
    vent_data = vent_data.reset_index()

    # Get the patients without the intervention in there too so that we
    ids_with = vent_data['stay_id']
    ids_with = set(map(int, ids_with))
    ids_all = set(map(int, icuids_to_keep))
    ids_without = (ids_all - ids_with)
    #ids_without = map(int, ids_without)

    # Create a new fake dataframe with blanks on all vent entries
    out_data = data.copy(deep=True)
    out_data = out_data.reset_index()
    out_data = out_data.set_index('stay_id')
    out_data = out_data.iloc[out_data.index.isin(ids_without)]
    out_data = out_data.reset_index()
    out_data = out_data[['subject_id', 'hadm_id', 'stay_id']]
    out_data['max_hours'] = out_data['stay_id'].map(icustay_timediff)

    # Create all 0 column for vent
    out_data = out_data.groupby('stay_id')
    out_data = out_data.apply(add_blank_indicators)
    out_data.rename(columns = {'on':'vent'}, inplace=True)
    out_data = out_data.reset_index()

    # Concatenate all the data vertically
    Y = pd.concat([vent_data[['subject_id', 'hadm_id', 'stay_id', 'hours_in', 'vent']],
                   out_data[['subject_id', 'hadm_id', 'stay_id', 'hours_in', 'vent']]],
                  axis=0)

    # Start merging all other interventions
    table_names = [
        #'vasopressor',
        #'adenosine_durations',
        'dobutamine',
        'dopamine',
        'epinephrine',
        #'isuprel_durations',
        'milrinone',
        'norepinephrine',
        'phenylephrine',
        'vasopressin'
    ]
    # column_names = ['vaso', 'adenosine', 'dobutamine', 'dopamine', 'epinephrine', 'isuprel', 
    #                 'milrinone', 'norepinephrine', 'phenylephrine', 'vasopressin']
    column_names = ['dobutamine', 'dopamine', 'epinephrine', 
                    'milrinone', 'norepinephrine', 'phenylephrine', 'vasopressin']

    # TODO(mmd): This section doesn't work. What is its purpose?
    for t, c in zip(table_names, column_names):
        # TOTAL VASOPRESSOR DATA
        query = """
        select i.subject_id, i.hadm_id, v.stay_id, v.vasonum, v.starttime, v.endtime
        FROM icustay_detail i
        INNER JOIN {table} v ON i.stay_id = v.stay_id
        where v.stay_id in ({icuids})
        and v.starttime between intime and outtime
        and v.endtime between intime and outtime;
        """
        query = """
        select i.subject_id, i.hadm_id, v.stay_id, v.starttime, v.endtime
        FROM icustay_detail i
        INNER JOIN {table} v ON i.stay_id = v.stay_id
        where v.stay_id in ({icuids})
        and v.starttime between i.icu_intime and i.icu_outtime
        and v.endtime between i.icu_intime and i.icu_outtime;
        """
        new_data = querier.query(query_string=query, extra_template_vars=dict(table=t))
        # get vasonum
        vent_data.loc[:, 'vasonum'] = new_data.groupby(['stay_id']).cumcount()
        new_data = continuous_outcome_processing(new_data, data, icustay_timediff)
        new_data = new_data.apply(add_outcome_indicators)
        new_data.rename(columns={'on': c}, inplace=True)
        new_data = new_data.reset_index()
        # c may not be in Y if we are only extracting a subset of the population, in which c was never
        # performed.
        if not c in new_data:
            print("Column ", c, " not in data.")
            continue

        Y = Y.merge(
            new_data[['subject_id', 'hadm_id', 'stay_id', 'hours_in', c]],
            on=['subject_id', 'hadm_id', 'stay_id', 'hours_in'],
            how='left'
        )

        # Sort the values
        Y.fillna(0, inplace=True)
        Y[c] = Y[c].astype(int)
        #Y = Y.sort_values(['subject_id', 'stay_id', 'hours_in']) #.merge(df3,on='name')
        Y = Y.reset_index(drop=True)
        print('Extracted ' + c + ' from ' + t)


    tasks=["colloid_bolus", "crystalloid_bolus", "nivdurations"]

    tasks =[('crystalloid', 'crystalloid_bolus'),('colloids','colloid_bolus'), ('non iv', 'nivdurations')]

    for task, task_rename in tasks:
        # if task=='nivdurations':
        #     query = """
        #     select i.subject_id, i.hadm_id, v.stay_id, v.starttime, v.endtime
        #     FROM icustay_detail i
        #     INNER JOIN {table} v ON i.stay_id = v.stay_id
        #     where v.stay_id in ({icuids})
        #     and v.starttime between intime and outtime
        #     and v.endtime between intime and outtime;
        #     """
        # else:
        #     query = """
        #     select i.subject_id, i.hadm_id, v.stay_id, v.charttime AS starttime, 
        #            v.charttime AS endtime
        #     FROM icustay_detail i
        #     INNER JOIN {table} v ON i.stay_id = v.stay_id
        #     where v.stay_id in ({icuids})
        #     and v.charttime between iintime and outtime
        #     """

        query = """
        select i.subject_id, i.hadm_id, v.stay_id, v.starttime, v.endtime
        FROM icustay_detail i
        INNER JOIN mimiciv_icu.inputevents v ON i.stay_id = v.stay_id
        where v.stay_id in ({icuids}) 
        and LOWER(v.ordercategoryname) LIKE '%{table}%'
        AND v.starttime between i.icu_intime and i.icu_outtime
        and v.endtime between i.icu_intime and i.icu_outtime;
        """

        new_data = querier.query(query_string=query, extra_template_vars=dict(table=task))
        if new_data.shape[0] == 0: continue
        new_data = continuous_outcome_processing(new_data, data, icustay_timediff)
        new_data = new_data.apply(add_outcome_indicators)
        new_data.rename(columns = {'on':task_rename}, inplace=True)
        new_data = new_data.reset_index()
        Y = Y.merge(
            new_data[['subject_id', 'hadm_id', 'stay_id', 'hours_in', task_rename]],
            on=['subject_id', 'hadm_id', 'stay_id', 'hours_in'],
            how='left'
        )

        # Sort the values
        Y.fillna(0, inplace=True)
        Y[task_rename] = Y[task_rename].astype(int)
        Y = Y.reset_index(drop=True)
        print('Extracted ' + task_rename)


    # TODO: ADD THE RBC/PLT/PLASMA DATA
    # TODO: ADD DIALYSIS DATA
    # TODO: ADD INFECTION DATA
    # TODO: Move queries to files
    querier.exclusion_criteria_template_vars = old_template_vars

    Y = Y.filter(items=['subject_id', 'hadm_id', 'stay_id', 'hours_in', 'vent'] + column_names + tasks)
    Y.subject_id = Y.subject_id.astype(int)
    Y.stay_id = Y.stay_id.astype(int)
    Y.hours_in = Y.hours_in.astype(int)
    Y.vent = Y.vent.astype(int)
    # Y.vaso = Y.vaso.astype(int)
    y_id_cols = ID_COLS + ['hours_in']
    Y = Y.sort_values(y_id_cols)
    Y.set_index(y_id_cols, inplace=True)

    print('Shape of Y : ', Y.shape)


    # SAVE AS NUMPY ARRAYS AND TEXT FILES
    #np_Y = Y.as_matrix()
    #np.save(os.path.join(outPath, outcome_filename), np_Y)


    # Turn back into columns
    df = Y.reset_index()
    df = sanitize_df(df, outcome_schema) 
    csv_fpath = os.path.join(outPath, outcome_filename)
    save_sanitized_df_to_csv(csv_fpath, df, outcome_schema)


    col_names  = list(df.columns.values)
    col_names = col_names[3:]
    with open(os.path.join(outPath, outcome_columns_filename), 'w') as f:
        f.write('\n'.join(col_names))

    print('a',df.columns.names)

    # TODO(mmd): Why does df have the index? Is sanitize making multiindex?
    # SAVE THE DATA AS A PANDAS OBJECT
    # TODO(mike hughes): Why writing out Y after you've separately sanitized df?
    Y.to_hdf(os.path.join(outPath, outcome_hd5_filename), 'Y')
    return df

# Apply the variable limits to remove things
# TODO(mmd): controlled printing.
def apply_variable_limits(df, var_ranges, var_names_index_col='LEVEL2'):
    idx_vals        = df.index.get_level_values(var_names_index_col)
    non_null_idx    = ~df.value.isnull()
    var_names       = set(idx_vals)
    var_range_names = set(var_ranges.index.values)

    for var_name in var_names:
        var_name_lower = var_name.lower()
        if var_name_lower not in var_range_names:
            print("No known ranges for %s" % var_name)
            continue

        outlier_low_val, outlier_high_val, valid_low_val, valid_high_val = [
            var_ranges.loc[var_name_lower, x] for x in ('OUTLIER_LOW','OUTLIER_HIGH','VALID_LOW','VALID_HIGH')
        ]

        running_idx = non_null_idx & (idx_vals == var_name)

        outlier_low_idx  = (df.value < outlier_low_val)
        outlier_high_idx = (df.value > outlier_high_val)
        valid_low_idx    = ~outlier_low_idx & (df.value < valid_low_val)
        valid_high_idx   = ~outlier_high_idx & (df.value > valid_high_val)

        var_outlier_idx   = running_idx & (outlier_low_idx | outlier_high_idx)
        var_valid_low_idx = running_idx & valid_low_idx
        var_valid_high_idx = running_idx & valid_high_idx

        df.loc[var_outlier_idx, 'value'] = np.nan
        df.loc[var_valid_low_idx, 'value'] = valid_low_val
        df.loc[var_valid_high_idx, 'value'] = valid_high_val

        n_outlier = sum(var_outlier_idx)
        n_valid_low = sum(var_valid_low_idx)
        n_valid_high = sum(var_valid_high_idx)
        if n_outlier + n_valid_low + n_valid_high > 0:
            print(
                "%s had %d / %d rows cleaned:\n"
                "  %d rows were strict outliers, set to np.nan\n"
                "  %d rows were low valid outliers, set to %.2f\n"
                "  %d rows were high valid outliers, set to %.2f\n"
                "" % (
                    var_name,
                    n_outlier + n_valid_low + n_valid_high, sum(running_idx),
                    n_outlier, n_valid_low, valid_low_val, n_valid_high, valid_high_val
                )
            )

    return df

def plot_variable_histograms(col_names, df):
    # Plot some of the data, just to make sure it looks ok
    for c, vals in df.iteritems():
        n = vals.dropna().count()
        if n < 2: continue

        # get median, variance, skewness
        med = vals.dropna().median()
        var = vals.dropna().var()
        skew = vals.dropna().skew()

        # plot
        fig = plt.figure(figsize=(13, 6))
        plt.subplots(figsize=(13,6))
        vals.dropna().plot.hist(bins=100, label='HIST (n={})'.format(n))

        # fake plots for KS test, median, etc
        plt.plot([], label=' ',color='lightgray')
        plt.plot([], label='Median: {}'.format(format(med,'.2f')),
                 color='lightgray')
        plt.plot([], label='Variance: {}'.format(format(var,'.2f')),
                 color='lightgray')
        plt.plot([], label='Skew: {}'.format(format(skew,'.2f')),
                 color='light:gray')

        # add title, labels etc.
        plt.title('{} measurements in ICU '.format(str(c)))
        plt.xlabel(str(c))
        plt.legend(loc="upper left", bbox_to_anchor=(1,1),fontsize=12)
        plt.xlim(0, vals.quantile(0.99))
        fig.savefig(os.path.join(outPath, (str(c) + '_HIST_.png')), bbox_inches='tight')

def add_exclusion_criteria_from_df(df, columns=[]):
    exclusion_criteria_template_vars.update({
        c: "','".join(
            set([str(v) for v in get_values_by_name_from_df_column_or_index(df, c)])
        ) for c in columns
    })

In [4]:
# Directly set all arguments as variables instead of using argparse
args = {
    'out_path': './output',
    'resource_path': os.path.expandvars("./resources/"),
    'queries_path': os.path.expandvars("./SQL_Queries/"),
    'extract_pop': 1,
    'extract_numerics': 1,
    'extract_outcomes': 1,
    'extract_codes': 1,
    'extract_notes': 1,
    'pop_size': 0,
    'exit_after_loading': 0,
    'var_limits': 1,
    'plot_hist': 1,
    'min_percent': 0.0,
    'min_age': 15,
    'min_duration': 12,
    'max_duration': 240
}

print("Running!")

# Print all arguments
for key in sorted(args.keys()):
    print(key, args[key])

# Validate resource path
if not isdir(args['resource_path']):
    raise ValueError("Invalid resource_path: %s" % args['resource_path'])

# Set filenames
mimic_mapping_filename = os.path.join(args['resource_path'], 'mimiciv_itemid_to_variable_map.csv')
range_filename = os.path.join(args['resource_path'], 'mimiciv_variable_ranges.csv')

Running!
exit_after_loading 0
extract_codes 1
extract_notes 1
extract_numerics 1
extract_outcomes 1
extract_pop 1
max_duration 240
min_age 15
min_duration 12
min_percent 0.0
out_path ./output
plot_hist 1
pop_size 0
queries_path ./SQL_Queries/
resource_path ./resources/
var_limits 1


In [5]:
# Load specs for output tables
static_data_schema = load_datapackage_schema(
    os.path.join(args['resource_path'], 'static_data_spec.json'))
outcome_data_schema = load_datapackage_schema(
    os.path.join(args['resource_path'], 'outcome_data_spec.json'))

if not isdir(args['out_path']):
    print('ERROR: OUTPATH %s DOES NOT EXIST' % args['out_path'])
    sys.exit()
else:
    outPath = args['out_path']

In [6]:
# Modify the filenames
if args['pop_size'] > 0:
    pop_size = str(args['pop_size'])

    static_filename = splitext(static_filename)[0] + '_' + pop_size + splitext(static_filename)[1]
    dynamic_filename = splitext(dynamic_filename)[0] + '_' + pop_size + splitext(dynamic_filename)[1]
    #columns_filename = splitext(columns_filename)[0] + '_' + pop_size + splitext(columns_filename)[1]
    subjects_filename = splitext(subjects_filename)[0] + '_' + pop_size + splitext(subjects_filename)[1]
    times_filename = splitext(times_filename)[0] + '_' + pop_size + splitext(times_filename)[1]
    dynamic_hd5_filename = splitext(dynamic_hd5_filename)[0] + '_' + pop_size + splitext(dynamic_hd5_filename)[1]
    outcome_filename = splitext(outcome_filename)[0] + '_' + pop_size + splitext(outcome_filename)[1]
    dynamic_hd5_filt_filename = splitext(dynamic_hd5_filt_filename)[0] + '_' + pop_size + splitext(dynamic_hd5_filt_filename)[1]
    outcome_hd5_filename = splitext(outcome_hd5_filename)[0] + '_' + pop_size + splitext(outcome_hd5_filename)[1]
    #outcome_columns_filename = splitext(outcome_columns_filename)[0] + '_' + pop_size + splitext(outcome_columns_filename)[1]
    codes_hd5_filename = splitext(codes_hd5_filename)[0] + '_' + pop_size + splitext(codes_hd5_filename)[1]
    notes_hd5_filename = splitext(notes_hd5_filename)[0] + '_' + pop_size + splitext(notes_hd5_filename)[1]
    idx_hd5_filename = splitext(idx_hd5_filename)[0] + '_' + pop_size + splitext(idx_hd5_filename)[1]

# dbname = args['psql_dbname']
# schema_name = args['psql_schema_name']
# query_args = {'dbname': dbname}
# if args['psql_host'] is not None: query_args['host'] = args['psql_host']
# if args['psql_user'] is not None: query_args['user'] = args['psql_user']
# if args['psql_password'] is not None: query_args['password'] = args['psql_password']

In [7]:
# querier = MIMIC_Querier(query_args=query_args, schema_name=schema_name)

#############
# Population extraction

data = None
if (args['extract_pop'] == 0 | (args['extract_pop'] == 1) ) & isfile(os.path.join(outPath, static_filename)):
    print("Reloading data from %s" % os.path.join(outPath, static_filename))
    data = pd.read_csv(os.path.join(outPath, static_filename))
    data = sanitize_df(data, static_data_schema)
elif (args['extract_pop'] == 1 & (not isfile(os.path.join(outPath, static_filename)))) | (args['extract_pop'] == 2):
    print("Building data from scratch.")
    pop_size_string = ''
    if args['pop_size'] > 0:
        pop_size_string = 'LIMIT ' + str(args['pop_size'])

    min_age_string = str(args['min_age'])
    min_dur_string = str(args['min_duration'])
    max_dur_string = str(args['max_duration'])
    min_day_string = str(float(args['min_duration'])/24)

    template_vars = dict(
        limit=pop_size_string, min_age=min_age_string, min_dur=min_dur_string, max_dur=max_dur_string,
        min_day=min_day_string
    )

    # data_df = querier.query(query_file=STATICS_QUERY_PATH, extra_template_vars=template_vars)
    data_df = pd.read_csv("./SQL_Queries/statics_bq.csv")
    print(data_df.head())
    data_df = sanitize_df(data_df, static_data_schema)
    print(2, data_df.head())

    print("Storing data @ %s" % os.path.join(outPath, static_filename))
    data = save_pop(data_df, outPath, static_filename, args['pop_size'], static_data_schema)
    print(3, data.head())
    
if data is None: print('SKIPPED static_data')
else:
    # So all subsequent queries will limit to just that already extracted in data_df.
    add_exclusion_criteria_from_df(data, columns=['hadm_id', 'subject_id'])
    # querier.add_exclusion_criteria_from_df(data, columns=['hadm_id', 'subject_id'])
    print("loaded static_data")

Reloading data from ./output/static_data.csv
loaded static_data


In [8]:
#############
# If there is numerics extraction
import pickle


X = None
if (args['extract_numerics'] == 0 | (args['extract_numerics'] == 1) ) & isfile(os.path.join(outPath, dynamic_hd5_filename)):
    print("Reloading X from %s" % os.path.join(outPath, dynamic_hd5_filename))
    X = pd.read_hdf(os.path.join(outPath, dynamic_hd5_filename))
elif (args['extract_numerics'] == 1 & (not isfile(os.path.join(outPath, dynamic_hd5_filename)))) | (args['extract_numerics'] == 2):
    print("Extracting vitals data...")
    start_time = time.time()

    ########
    # Step 1) Get the set of variables we want for the patients we've identified!
    icuids_to_keep = get_values_by_name_from_df_column_or_index(data, 'stay_id')
    icuids_to_keep = set([str(s) for s in icuids_to_keep])
    data = data.copy(deep=True).reset_index().set_index('stay_id')

    # Select out SID, TIME, ITEMID, VALUE form each of the sources!
    var_map = get_variable_mapping(mimic_mapping_filename)
    var_ranges = get_variable_ranges(range_filename)

    chartitems_to_keep = var_map.loc[var_map['linksto'] == 'chartevents'].ITEMID
    chartitems_to_keep = set([ str(i) for i in chartitems_to_keep ])


    labitems_to_keep = var_map.loc[var_map['linksto'] == 'mimiciv_hosp.labevents'].ITEMID
    labitems_to_keep = set([ str(i) for i in labitems_to_keep ])

    # TODO(mmd): Use querier, move to file
    # con = psycopg2.connect(**query_args)
    # cur = con.cursor()

    print("  starting db query with %d subjects..." % (len(icuids_to_keep)))
    # cur.execute('SET search_path to ' + schema_name)


    # query = \
    # """
    # select c.subject_id, i.hadm_id, c.stay_id, c.charttime, c.itemid, c.value, valueuom
    # FROM icustay_detail i
    # INNER JOIN chartevents c ON i.stay_id = c.stay_id
    # where c.stay_id in ({icuids})
    #     and c.itemid in ({chitem})
    #     and c.charttime between i.icu_intime and i.icu_outtime
    #     and c.warning is distinct from 1
    #     and c.valuenum is not null

    # UNION ALL

    # select distinct i.subject_id, i.hadm_id, i.stay_id, l.charttime, l.itemid, l.value, valueuom
    # FROM icustay_detail i
    # INNER JOIN labevents l ON i.hadm_id = l.hadm_id
    # where i.stay_id in ({icuids})
    #     and l.itemid in ({lbitem})
    #     and l.charttime between (i.icu_intime - interval '6' hour) and i.icu_outtime
    #     and l.valuenum > 0 -- lab values cannot be 0 and cannot be negative
    # ;
    # """.format(icuids=','.join(icuids_to_keep), chitem=','.join(chartitems_to_keep), lbitem=','.join(labitems_to_keep))
    # X = pd.read_sql_query(query, con)
    X = pd.read_parquet('SQL_Queries/charttime.parquet')
    # X = X.set_index(['stay_id', 'itemid'])

    # itemids = set(X.itemid.astype(str))

    # query_d_items = \
    # """
    # SELECT itemid, label, linksto, category, unitname
    # FROM d_items
    # WHERE itemid in ({itemids})
    # ;
    # """.format(itemids=','.join(itemids))
    # I = pd.read_sql_query(query_d_items, con).set_index('itemid')
    with open("SQL_Queries/itemid_label.pkl", 'rb') as f:
        I = pickle.load(f)
        I = I.set_index('itemid')

    # cur.close()
    # con.close()
    print("  db query finished after %.3f sec" % (time.time() - start_time))
    

Extracting vitals data...
  starting db query with 46103 subjects...
  db query finished after 5.120 sec


In [9]:
X.head()

,subject_id,hadm_id,stay_id,charttime,itemid,value,valueuom
0,10000980,26913865,39765666,2189-06-27 08:54:00,220210,23.0,insp/min
1,10000980,26913865,39765666,2189-06-27 08:55:00,220179,150.0,mmHg
2,10000980,26913865,39765666,2189-06-27 08:55:00,220180,77.0,mmHg
3,10000980,26913865,39765666,2189-06-27 08:55:00,220181,92.0,mmHg
4,10000980,26913865,39765666,2189-06-27 08:56:00,220277,100.0,%


In [10]:
# Check DataFrame structures
print("X index:", X.index.names)
print("I index:", I.index.names)
print("X columns:", X.columns.tolist())
print("I columns:", I.columns.tolist())

X index: [None]
I index: ['itemid']
X columns: ['subject_id', 'hadm_id', 'stay_id', 'charttime', 'itemid', 'value', 'valueuom']
I columns: ['label']


In [82]:
# X = save_numerics(
#         data, X, I, var_map, var_ranges, outPath, dynamic_filename, columns_filename, subjects_filename,
#         times_filename, dynamic_hd5_filename, apply_var_limit=args['var_limits'],
#         min_percent=args['min_percent']
#     )

var map    itemid                                label  fluid   category   count  min  \
0   50801           Alveolar-arterial Gradient  Blood  Blood Gas   12558  ___   
1   50802                          Base Excess  Blood  Blood Gas  511502  ___   
2   50803  Calculated Bicarbonate, Whole Blood  Blood  Blood Gas   23226  ___   
3   50804                 Calculated Total CO2  Blood  Blood Gas  511464  ___   
4   50805                    Carboxyhemoglobin  Blood  Blood Gas    5626  ___   

   max                 linksto param_type unitname  ITEMID  
0   99  mimiciv_hosp.labevents        NaN    mm Hg   50801  
1   -9  mimiciv_hosp.labevents        NaN    mEq/L   50802  
2   98  mimiciv_hosp.labevents        NaN    mEq/L   50803  
3    9  mimiciv_hosp.labevents        NaN    mEq/L   50804  
4  9.8  mimiciv_hosp.labevents        NaN        %   50805  
1    subject_id   hadm_id   stay_id           charttime  itemid  value  valueuom
0    10000980  26913865  39765666 2189-06-27 08:54:00  220

: 

In [11]:
def process_numerics_chunk(X, data, chunk_size=10000):
    """Process X data in chunks to avoid memory issues"""
    chunks = []
    for start_idx in range(0, len(X), chunk_size):
        chunk = X.iloc[start_idx:start_idx + chunk_size].copy()
        
        # Convert to numeric and handle time
        chunk['value'] = pd.to_numeric(chunk['value'], 'coerce')
        
        # Calculate hours_in
        chunk = chunk.join(data[['intime']])
        charttime = chunk['charttime'].astype(int) / 10**9
        intime = chunk['intime'].astype(int) / 10**9
        chunk['hours_in'] = (charttime - intime)//3600
        
        chunk.drop(columns=['charttime', 'intime'], inplace=True)
        chunks.append(chunk)
        
    return pd.concat(chunks)

def join_variable_mapping(X, var_map, I):
    """Join X with variable mapping and item information"""
    try:
        # Reset indexes for clean joins
        X = X.reset_index()
        var_map = var_map.reset_index()
        I = I.reset_index()
        
        # Join with var_map first
        X = pd.merge(X, var_map, on='itemid', how='left')
        
        # Join with I
        X = pd.merge(X, I[['itemid', 'label']], on='itemid', how='left')
        
        return X
    except Exception as e:
        print(f"Error in join_variable_mapping: {e}")
        return None

def safe_save_numerics(data, X, I, var_map, var_ranges, outPath, 
                      dynamic_filename, columns_filename, subjects_filename,
                      times_filename, dynamic_hd5_filename, 
                      apply_var_limit, min_percent):
    """Safer version of save_numerics with error handling"""
    try:
        # Process in chunks
        X_processed = process_numerics_chunk(X, data)
        
        # Join with mappings
        X_processed = join_variable_mapping(X_processed, var_map, I)
        
        # Set multi-index
        index_cols = ['subject_id', 'hadm_id', 'stay_id', 'itemid']
        X_processed = X_processed.set_index(index_cols)
        
        # Standardize units if needed
        if apply_var_limit:
            X_processed = standardize_units(X_processed, name_col='label')
        
        # Save to HDF5
        X_processed.to_hdf(os.path.join(outPath, dynamic_hd5_filename), 'vitals_labs')
        
        return X_processed
        
    except Exception as e:
        print(f"Error in safe_save_numerics: {e}")
        return None

In [12]:
# Usage
X_processed = safe_save_numerics(
    data, X, I, var_map, var_ranges, outPath, 
    dynamic_filename, columns_filename, subjects_filename,
    times_filename, dynamic_hd5_filename, 
    apply_var_limit=args['var_limits'],
    min_percent=args['min_percent']
)

Error in safe_save_numerics: 'label'


In [13]:
def safe_save_numerics_v2(data, X, I, var_map, var_ranges, outPath, 
                         dynamic_filename, columns_filename, subjects_filename,
                         times_filename, dynamic_hd5_filename, 
                         apply_var_limit, min_percent):
    try:
        # 1. First check dataframes
        print("Initial shapes:")
        print(f"X shape: {X.shape}, columns: {X.columns.tolist()}")
        print(f"I shape: {I.shape}, columns: {I.columns.tolist()}")
        print(f"var_map shape: {var_map.shape}, columns: {var_map.columns.tolist()}")
        
        # 2. Process X in chunks
        X_processed = X.copy()
        X_processed['value'] = pd.to_numeric(X_processed['value'], 'coerce')
        
        # 3. Join with intime and calculate hours
        X_processed = X_processed.set_index('stay_id').join(data[['intime']])
        X_processed['hours_in'] = (pd.to_datetime(X_processed['charttime']).astype(np.int64) - 
                                 pd.to_datetime(X_processed['intime']).astype(np.int64)) // 3600000000000
        
        # 4. Drop unnecessary columns
        X_processed = X_processed.drop(columns=['charttime', 'intime'])
        
        # 5. Join with var_map and I
        X_processed = X_processed.reset_index()
        var_map_subset = var_map[['itemid', 'label', 'category']].copy()
        X_processed = pd.merge(X_processed, var_map_subset, on='itemid', how='left')
        
        # 6. Set multi-index
        index_cols = ['subject_id', 'hadm_id', 'stay_id', 'itemid']
        X_processed = X_processed.set_index(index_cols)
        
        # 7. Standardize units if needed
        if apply_var_limit and 'label' in X_processed.columns:
            X_processed = standardize_units(X_processed, name_col='label')
        
        # 8. Group and aggregate
        agg_cols = ['value']
        X_processed = X_processed.groupby(index_cols + ['hours_in'])[agg_cols].agg(['mean', 'std', 'count'])
        
        # 9. Save to HDF5
        X_processed.to_hdf(os.path.join(outPath, dynamic_hd5_filename), 'vitals_labs')
        
        return X_processed
        
    except Exception as e:
        print(f"Error in safe_save_numerics_v2: {e}")
        print(f"Error location: {e.__traceback__.tb_lineno}")
        return None

# Usage
X_processed = safe_save_numerics_v2(
    data, X, I, var_map, var_ranges, outPath, 
    dynamic_filename, columns_filename, subjects_filename,
    times_filename, dynamic_hd5_filename, 
    apply_var_limit=args['var_limits'],
    min_percent=args['min_percent']
)

Initial shapes:
X shape: (59684393, 7), columns: ['subject_id', 'hadm_id', 'stay_id', 'charttime', 'itemid', 'value', 'valueuom']
I shape: (1411, 1), columns: ['label']
var_map shape: (4173, 11), columns: ['itemid', 'label', 'fluid', 'category', 'count', 'min', 'max', 'linksto', 'param_type', 'unitname', 'ITEMID']
Error in safe_save_numerics_v2: Saving a MultiIndex with an extension dtype is not supported.
Error location: 42


In [23]:
def safe_save_numerics_final(data, X, I, var_map, var_ranges, outPath, 
                         dynamic_filename, columns_filename, subjects_filename,
                         times_filename, dynamic_hd5_filename, 
                         apply_var_limit, min_percent, chunk_size=100000):
    try:
        print("Starting processing of full dataset...")
        print(f"Total rows to process: {len(X)}")
        
        # Process in chunks
        chunks = []
        for start_idx in range(0, len(X), chunk_size):
            end_idx = min(start_idx + chunk_size, len(X))
            print(f"Processing chunk {start_idx//chunk_size + 1}, rows {start_idx} to {end_idx}")
            
            # 1. Get chunk
            X_chunk = X.iloc[start_idx:end_idx].copy()
            
            # 2. Process values
            X_chunk['value'] = pd.to_numeric(X_chunk['value'], errors='coerce')
            X_chunk['itemid'] = pd.to_numeric(X_chunk['itemid'], errors='coerce')
            
            # 3. Calculate hours
            X_chunk = X_chunk.set_index('stay_id').join(data[['intime']])
            X_chunk['charttime'] = pd.to_datetime(X_chunk['charttime'])
            X_chunk['intime'] = pd.to_datetime(X_chunk['intime'])
            X_chunk['hours_in'] = ((X_chunk['charttime'] - X_chunk['intime'])
                               .dt.total_seconds() / 3600)
            
            # 4. Clean and convert
            X_chunk = X_chunk.drop(columns=['charttime', 'intime']).reset_index()
            X_chunk = X_chunk.astype({
                'subject_id': 'int32',
                'hadm_id': 'int32',
                'stay_id': 'int32',
                'itemid': 'int32',
                'value': 'float32',
                'hours_in': 'float32'
            })
            
            # 5. Group and aggregate
            group_cols = ['subject_id', 'hadm_id', 'stay_id', 'itemid', 'hours_in']
            chunk_agg = X_chunk.groupby(group_cols)['value'].agg(['mean', 'std', 'count'])
            chunks.append(chunk_agg)
            
            # Clear memory
            del X_chunk
            
        # Combine all chunks
        print("Combining chunks...")
        result = pd.concat(chunks)
        
        # Final aggregation
        print("Final aggregation...")
        result = result.groupby(level=group_cols).agg({
            'mean': 'mean',
            'std': 'mean',  # Take mean of standard deviations
            'count': 'sum'
        })
        
        # Save to HDF5
        print("Saving to HDF5...")
        result.to_hdf(
            os.path.join(outPath, dynamic_hd5_filename),
            'vitals_labs',
            format='table',
            mode='w'
        )
        
        print("Processing complete!")
        return result
        
    except Exception as e:
        print(f"Error in safe_save_numerics_final: {e}")
        print(f"Error location: {e.__traceback__.tb_lineno}")
        return None

# Process full dataset
X_processed = safe_save_numerics_final(
    data, X, I, var_map, var_ranges, outPath, 
    dynamic_filename, columns_filename, subjects_filename,
    times_filename, dynamic_hd5_filename, 
    apply_var_limit=args['var_limits'],
    min_percent=args['min_percent']
)

Starting processing of full dataset...
Total rows to process: 59684393
Processing chunk 1, rows 0 to 100000
Processing chunk 2, rows 100000 to 200000
Processing chunk 3, rows 200000 to 300000
Processing chunk 4, rows 300000 to 400000
Processing chunk 5, rows 400000 to 500000
Processing chunk 6, rows 500000 to 600000
Processing chunk 7, rows 600000 to 700000
Processing chunk 8, rows 700000 to 800000
Processing chunk 9, rows 800000 to 900000
Processing chunk 10, rows 900000 to 1000000
Processing chunk 11, rows 1000000 to 1100000
Processing chunk 12, rows 1100000 to 1200000
Processing chunk 13, rows 1200000 to 1300000
Processing chunk 14, rows 1300000 to 1400000
Processing chunk 15, rows 1400000 to 1500000
Processing chunk 16, rows 1500000 to 1600000
Processing chunk 17, rows 1600000 to 1700000
Processing chunk 18, rows 1700000 to 1800000
Processing chunk 19, rows 1800000 to 1900000
Processing chunk 20, rows 1900000 to 2000000
Processing chunk 21, rows 2000000 to 2100000
Processing chunk 

In [24]:
def save_numerics_chunked(data, X, I, var_map, var_ranges, outPath, 
                         dynamic_filename, columns_filename, subjects_filename,
                         times_filename, dynamic_hd5_filename, 
                         apply_var_limit, min_percent, chunk_size=100000):
    """Memory-efficient version of save_numerics"""
    try:
        assert len(data) > 0 and len(X) > 0, "Must provide some input data to process."
        print('Processing data in chunks...')

        # 1. Initial Setup
        var_map = var_map.groupby('itemid').last()
        chunks = []
        
        # 2. Process in chunks
        for start_idx in range(0, len(X), chunk_size):
            end_idx = min(start_idx + chunk_size, len(X))
            print(f"Processing rows {start_idx} to {end_idx}")
            
            X_chunk = X.iloc[start_idx:end_idx].copy()
            
            # Basic processing
            X_chunk['value'] = pd.to_numeric(X_chunk['value'], 'coerce')
            for col in ID_COLS:
                if col in X_chunk.columns:
                    X_chunk.loc[:,col] = X_chunk[col].astype(int)
            
            # Calculate hours_in
            X_chunk = X_chunk.set_index('stay_id').join(data[['intime']])
            charttime = X_chunk['charttime'].astype(int) / 10**9
            intime = X_chunk['intime'].astype(int) / 10**9
            X_chunk['hours_in'] = (charttime - intime)//3600
            
            X_chunk.drop(columns=['charttime', 'intime'], inplace=True)
            X_chunk.set_index('itemid', append=True, inplace=True)
            
            # Join with mappings
            X_chunk = X_chunk.join(var_map)
            X_chunk = X_chunk.join(I, rsuffix='_remove').set_index(['label'], append=True)
            
            # Standardize units
            standardize_units(X_chunk, name_col='label', inplace=True)
            
            if apply_var_limit > 0:
                X_chunk = apply_variable_limits(X_chunk, var_ranges, 'label')
            
            chunks.append(X_chunk)
            
        # 3. Combine chunks
        print("Combining chunks...")
        X = pd.concat(chunks)
        del chunks
        
        # 4. Group and aggregate
        X = X.drop(columns=['count', 'ITEMID'])
        X = X.groupby(ID_COLS + ITEM_COLS + ['hours_in']).agg(['mean', 'std', 'count'])
        X.columns = X.columns.droplevel(0)
        X.columns.names = ['Aggregation Function']
        
        # 5. Process missing hours
        data['max_hours'] = (data['outtime'] - data['intime']).apply(lambda x: max(0, x.days*24 + x.seconds // 3600))
        missing_hours_fill = range_unnest(data, 'max_hours', out_col_name='hours_in', reset_index=True)
        missing_hours_fill['tmp'] = np.NaN
        
        fill_df = data.reset_index()[ID_COLS].join(missing_hours_fill.set_index('stay_id'), on='stay_id')
        fill_df.set_index(ID_COLS + ['hours_in'], inplace=True)
        
        # 6. Final processing
        X = X.unstack(level=ITEM_COLS)
        X.columns = X.columns.reorder_levels(order=ITEM_COLS + ['Aggregation Function'])
        X = X.reindex(fill_df.index)
        X = X.sort_index(axis=0).sort_index(axis=1)
        
        # 7. Handle missing values
        idx = pd.IndexSlice
        X.loc[:, idx[:,:, 'count']] = X.loc[:, idx[:,:, 'count']].fillna(0)
        
        # 8. Drop sparse columns
        if min_percent > 0:
            n = round((1-min_percent/100.0)*X.shape[0])
            drop_col = []
            for k in X.columns:
                if k[-1] == 'mean' and X[k].isnull().sum() > n:
                    drop_col.append(k[:-1])
            X = X.drop(columns=drop_col)
        
        # 9. Save outputs
        if dynamic_filename:
            np.save(os.path.join(outPath, dynamic_filename), X.as_matrix())
        if dynamic_hd5_filename:
            X.to_hdf(os.path.join(outPath, dynamic_hd5_filename), 'X')
        if columns_filename:
            col_names = [str(x) for x in X.columns.values]
            with open(os.path.join(outPath, columns_filename), 'w') as f:
                f.write('\n'.join(col_names))
        if subjects_filename:
            np.save(os.path.join(outPath, subjects_filename), data['subject_id'].as_matrix())
        if times_filename:
            np.save(os.path.join(outPath, times_filename), data['max_hours'].as_matrix())
            
        return X
        
    except Exception as e:
        print(f"Error in save_numerics_chunked: {e}")
        print(f"Error location: {e.__traceback__.tb_lineno}")
        return None

In [ ]:
X_processed = save_numerics_chunked(
    data=data, 
    X=X, 
    I=I, 
    var_map=var_map, 
    var_ranges=var_ranges,
    outPath="test_output",
    dynamic_filename=dynamic_filename,
    columns_filename=columns_filename,
    subjects_filename=subjects_filename,
    times_filename=times_filename,
    dynamic_hd5_filename=dynamic_hd5_filename,
    apply_var_limit=args['var_limits'],
    min_percent=args['min_percent'],
    chunk_size=100000  # Adjust based on available RAM
)

In [25]:
def create_test_subset(X, data, var_map, I, n_patients=10):
    """Create small test dataset"""
    # Get sample subjects
    test_subjects = X['subject_id'].unique()[:n_patients]
    
    # Filter dataframes
    X_test = X[X['subject_id'].isin(test_subjects)].copy()
    data_test = data[data['subject_id'].isin(test_subjects)].copy()
    
    # Get relevant itemids
    test_itemids = X_test['itemid'].unique()
    var_map_test = var_map[var_map['itemid'].isin(test_itemids)].copy()
    I_test = I[I.index.isin(test_itemids)].copy()
    
    return X_test, data_test, var_map_test, I_test

# Test usage
def test_save_numerics():
    # Create test directory
    test_output_path = 'test_output'
    os.makedirs(test_output_path, exist_ok=True)
    
    # Create test subset
    X_test, data_test, var_map_test, I_test = create_test_subset(X, data, var_map, I)
    
    # Run save_numerics_chunked with test data
    X_processed = save_numerics_chunked(
        data=data_test,
        X=X_test,
        I=I_test,
        var_map=var_map_test,
        var_ranges=var_ranges,
        outPath=test_output_path,
        dynamic_filename='test_dynamic.npy',
        columns_filename='test_columns.txt',
        subjects_filename='test_subjects.npy',
        times_filename='test_times.npy',
        dynamic_hd5_filename='test_dynamic.h5',
        apply_var_limit=False,
        min_percent=0,
        chunk_size=1000
    )
    
    print(f"Test output shape: {X_processed.shape}")
    return X_processed

# Run test
if __name__ == "__main__":
    test_result = test_save_numerics()

Processing data in chunks...
Processing rows 0 to 1000
Processing rows 1000 to 2000
Processing rows 2000 to 3000
Processing rows 3000 to 4000
Processing rows 4000 to 5000
Processing rows 5000 to 6000
Processing rows 6000 to 7000
Processing rows 7000 to 8000
Processing rows 8000 to 9000
Processing rows 9000 to 10000
Processing rows 10000 to 11000
Processing rows 11000 to 12000
Processing rows 12000 to 13000
Processing rows 13000 to 14000
Processing rows 14000 to 15000
Processing rows 15000 to 16000
Processing rows 16000 to 17000
Processing rows 17000 to 18000
Processing rows 18000 to 18960
Combining chunks...
Error in save_numerics_chunked: Could not convert IU/L to numeric
Error location: 55


AttributeError: 'NoneType' object has no attribute 'shape'

In [19]:
def create_test_subset(X, data, var_map, I, n_patients=10):
    """Create small test dataset"""
    # Get sample patients
    test_subjects = X['subject_id'].unique()[:n_patients]
    
    # Filter all dataframes
    X_test = X[X['subject_id'].isin(test_subjects)].copy()
    data_test = data[data['subject_id'].isin(test_subjects)].copy()
    
    # Get relevant itemids
    test_itemids = X_test['itemid'].unique()
    var_map_test = var_map[var_map['itemid'].isin(test_itemids)].copy()
    I_test = I[I.index.isin(test_itemids)].copy()
    
    return X_test, data_test, var_map_test, I_test

def test_pipeline():
    # Load original data
    X = pd.read_parquet('SQL_Queries/charttime.parquet')
    with open("SQL_Queries/itemid_label.pkl", 'rb') as f:
        I = pickle.load(f)
    
    # Create test subset
    X_test, data_test, var_map_test, I_test = create_test_subset(X, data, var_map, I)
    
    # Test processing
    X_processed = safe_save_numerics_v3(
        data_test, X_test, I_test, var_map_test,
        var_ranges, 'test_output',
        'test_dynamic.csv', 'test_columns.csv',
        'test_subjects.csv', 'test_times.csv',
        'test_dynamic.h5',
        apply_var_limit=False,
        min_percent=0
    )
    
    return X_processed

# Run test
if __name__ == "__main__":
    test_pipeline()

Step 1: Initial Processing
Step 2: Calculating hours
Step 3: Cleaning columns
Step 4: Merging mappings
Step 5: Aggregating
Step 6: Converting dtypes
Step 7: Saving to HDF5
Error in safe_save_numerics_v3: cannot interpret dtype of [Int64]
Error location: 49


In [72]:
import pickle
first_time_flag = False
if first_time_flag:
    with open('./icuids_to_keep.pkl', 'wb') as f:
        pickle.dump(icuids_to_keep, f)
    with open('./chartitems_to_keep.pkl', 'wb') as f:
        pickle.dump(chartitems_to_keep, f)
    with open('./labitems_to_keep.pkl', 'wb') as f:
        pickle.dump(labitems_to_keep, f)
    # with open(filepath, 'rb') as f:
    #     return pickle.load(f)
    # filepath = './chartitems_to_keep.csv'
    # pd.Series(chartitems_to_keep).to_csv(filepath, index=False, header=['stay_id'])

In [ ]:


if X is None: print("SKIPPED vitals_hourly_data")
else:         print("LOADED vitals_hourly_data")

#############
# If there is codes extraction
C = None
if ( (args['extract_codes'] == 0) or (args['extract_codes'] == 1) ) and isfile(os.path.join(outPath, codes_hd5_filename)):
    print("Reloading codes from %s" % os.path.join(outPath, codes_hd5_filename))
    C = pd.read_hdf(os.path.join(outPath, codes_hd5_filename))
elif ( (args['extract_codes'] == 1) and (not isfile(os.path.join(outPath, codes_hd5_filename))) ) or (args['extract_codes'] == 2):
    print("Saving codes...")
    codes = querier.query(query_file=CODES_QUERY_PATH)
    C = save_icd_codes(codes, outPath, codes_hd5_filename)

if C is None: print("SKIPPED codes_data")
else:         print("LOADED codes_data")

#############
# If there is notes extraction
N = None
if ( (args['extract_notes'] == 0) or (args['extract_codes'] == 1) ) and isfile(os.path.join(outPath, notes_hd5_filename)):
    print("Reloading Notes.")
    N = pd.read_hdf(os.path.join(outPath, notes_hd5_filename))
elif ( (args['extract_notes'] == 1) and (not isfile(os.path.join(outPath, notes_hd5_filename))) ) or (args['extract_notes'] == 2):
    print("Saving notes...")
    notes = querier.query(query_file=NOTES_QUERY_PATH)
    N = save_notes(notes, outPath, notes_hd5_filename)

if N is None: print("SKIPPED notes_data")
else:         print("LOADED notes_data")

#############
# If there is outcome extraction
Y = None
if ( (args['extract_outcomes'] == 0) | (args['extract_outcomes'] == 1) ) & isfile(os.path.join(outPath, outcome_hd5_filename)):
    print("Reloading outcomes")
    Y = pd.read_hdf(os.path.join(outPath, outcome_hd5_filename))
elif ( (args['extract_outcomes'] == 1) & (not isfile(os.path.join(outPath, outcome_hd5_filename))) ) | (args['extract_outcomes'] == 2):
    print("Saving Outcomes...")
    Y = save_outcome(
        data, querier, outPath, outcome_filename, outcome_hd5_filename,
        outcome_columns_filename, outcome_data_schema, host=args['psql_host'],
    )


if X is not None: print("Numerics", X.shape, X.index.names, X.columns.names)
if Y is not None: print("Outcomes", Y.shape, Y.index.names, Y.columns.names, Y.columns)
if C is not None: print("Codes", C.shape, C.index.names, C.columns.names)
if N is not None: print("Notes", N.shape, N.index.names, N.columns.names)

# TODO(mmd): Do we want to align N like the others? Seems maybe wrong?

print(data.shape, data.index.names, data.columns.names)
if args['exit_after_loading']:
    sys.exit()


shared_idx = X.index
shared_sub = list(X.index.get_level_values('stay_id').unique())
#X = X.loc[shared_idx]
# TODO(mmd): Why does this work?
print(Y.head())
print(X.index.names, Y.index.names)
# get overlap of x.index and y.index
assert X.index.names==Y.index.names
shared_idx = X.index.intersection(Y.index)

Y = Y.loc[shared_idx]
# Problems start here.
if C is not None: C = C.loc[shared_idx]
data = data[data.index.get_level_values('stay_id').isin(set(shared_sub))]
data = data.reset_index().set_index(ID_COLS)


# Map the lowering function to all column names
X.columns = pd.MultiIndex.from_tuples(
    [tuple((str(l).lower() for l in cols)) for cols in X.columns], names=X.columns.names
)
# if args['group_by_level2']:
#     var_names = list(X.columns.get_level_values('LEVEL2'))
# else:
#     var_names = list(X.columns.get_level_values('itemid'))
var_names = list(X.columns.get_level_values('itemid'))


Y.columns = Y.columns.str.lower()
out_names = list(Y.columns.values[3:])
if C is not None:
    C.columns = C.columns.str.lower()
    icd_names = list(C.columns.values[1:])
data.columns = data.columns.str.lower()
static_names = list(data.columns.values[3:])


print('Shape of X : ', X.shape)
print('Shape of Y : ', Y.shape)
if C is not None: print('Shape of C : ', C.shape)
print('Shape of static : ', data.shape)
print('Variable names : ', ",".join(var_names))
print('Output names : ', ",".join(out_names))
if C is not None: print('Ic_dfD9 names : ', ",".join(icd_names))
print('Static data : ', ",".join(static_names))

X.to_hdf(os.path.join(outPath, dynamic_hd5_filt_filename), 'vitals_labs')
Y.to_hdf(os.path.join(outPath, dynamic_hd5_filt_filename), 'interventions')
if C is not None: C.to_hdf(os.path.join(outPath, dynamic_hd5_filt_filename), 'codes')
data.to_hdf(os.path.join(outPath, dynamic_hd5_filt_filename), 'patients', format='table')
#fencepost.to_hdf(os.path.join(outPath, dynamic_hd5_filt_filename), 'fencepost')

#############
#X.to_hdf(os.path.join(outPath, dynamic_hd5_filt_filename), 'X')
#print('FINISHED VAR LIMITS')

X_mean = X.iloc[:, X.columns.get_level_values(-1)=='mean']
X_mean.to_hdf(os.path.join(outPath, dynamic_hd5_filt_filename), 'vitals_labs_mean')

#TODO: Log the variables that are in 0-1 space, like
#to_log = ['fio2', 'glucose']
#for feat in to_log:
#    X[feat] = x[feat].apply(np.log)

#############
# Plot the histograms
if args['plot_hist'] == 1:
    plot_variable_histograms(var_names, X)

#############
# Print the total proportions!
rows, vars = X.shape
print('')
for l, vals in X.iteritems():
    ratio = 1.0 * vals.dropna().count() / rows
    print(str(l) + ': ' + str(round(ratio, 3)*100) + '% present')

#############
# Print the per subject proportions!
df = X.groupby(['subject_id']).count()
for k in [1, 2, 3]:
    print('% of subjects had at least ' + str(k) + ' present')
    d = df > k
    d = d.sum(axis=0)
    d = d / len(df)
    d = d.reset_index()
    for index, row in d.iterrows():
        print(str(index) + ': ' + str(round(row[0], 3)*100) + '%')
    print('\n')

print('Done!')